In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

In [6]:
# 自動下載並設定對應版本的 ChromeDriver
service = Service(ChromeDriverManager().install())

# 啟動瀏覽器 (不加入 headless 參數，讓我們能看到實體畫面)
driver = webdriver.Chrome(service=service)

# 將視窗最大化，確保 Google Maps 的左側資訊面板能完整顯示
driver.maximize_window()
print("瀏覽器啟動成功！")

瀏覽器啟動成功！


In [12]:
# [Cell 3] 前往絕對有數據的大地標測試
place_name = "新光三越 台南新天地"
search_query = place_name.replace(" ", "+")
url = f"https://www.google.com/maps/search/{search_query}"

print(f"正在前往：{url}")
driver.get(url)

# 等待網頁初始載入
time.sleep(4)

# 【新增魔法】讓網頁左側面板稍微往下滾動，確保圖表元素被載入 (Lazy loading 解法)
try:
    # 找尋左側面板的區塊並點擊一下，然後模擬按下 Page Down 往下滾
    pane = driver.find_element(By.XPATH, '//*[@role="main"]')
    pane.click()
    from selenium.webdriver.common.keys import Keys
    pane.send_keys(Keys.PAGE_DOWN)
    pane.send_keys(Keys.PAGE_DOWN)
except:
    pass

time.sleep(2) # 再等它一下
print("網頁載入且滾動完成！")

正在前往：https://www.google.com/maps/search/新光三越+台南新天地
網頁載入且滾動完成！


In [ ]:
# [Cell 4] 抓取熱門時段的隱藏文字並整理
print("開始尋找擁擠度標籤...")

# 把 XPath 條件放寬：只要 aria-label 裡面有「繁忙」、「busy」或「%」我們全都要！
xpath_query = "//*[contains(@aria-label, '繁忙') or contains(@aria-label, 'busy') or contains(@aria-label, '%')]"
elements = driver.find_elements(By.XPATH, xpath_query)

raw_crowd_data = []
for el in elements:
    label = el.get_attribute("aria-label")
    # 再次過濾，確保不是抓到其他無關的按鈕 (例如電池電量%)
    if label and ("繁忙" in label or "busy" in label or "通常" in label):
        raw_crowd_data.append(label)

print(f"成功抓取到 {len(raw_crowd_data)} 筆時段資料！")

if len(raw_crowd_data) > 0:
    print("\n資料預覽 (前 5 筆)：")
    for data in raw_crowd_data[:5]:
        print(data)
else:
    print("還是沒有抓到資料。請確認瀏覽器畫面中真的有長條圖出現！")

開始尋找擁擠度標籤...
成功抓取到 126 筆時段資料！

資料預覽 (前 5 筆)：
6時的繁忙程度通常為 0%。
7時的繁忙程度通常為 0%。
8時的繁忙程度通常為 0%。
9時的繁忙程度通常為 0%。
10時的繁忙程度通常為 0%。
